<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-25.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 25 - MapReduce (Teoría): Implementación desde cero

**Módulo:** Big Data Aplicado (BDA - 5075)

---

## Objetivos

- Implementar el paradigma MapReduce completo en Python puro
- Ejecutar Word Count paso a paso (Map → Shuffle → Reduce)
- Calcular estadísticas usando MapReduce (media, máximo)
- Visualizar el flujo de datos en cada fase

---

## 1. Framework MapReduce en Python

Construimos un mini-framework MapReduce que implementa las tres fases: Map, Shuffle/Sort y Reduce.

In [ ]:
from collections import defaultdict
from itertools import groupby
import time

class MapReduceFramework:
    """Mini-framework MapReduce en Python."""

    def __init__(self, mapper, reducer, combiner=None, verbose=False):
        self.mapper = mapper
        self.reducer = reducer
        self.combiner = combiner
        self.verbose = verbose

    def _fase_map(self, datos):
        """Fase MAP: aplica la función map a cada entrada."""
        pares_intermedios = []
        for i, (key, value) in enumerate(datos):
            resultado = self.mapper(key, value)
            pares_intermedios.extend(resultado)

        if self.verbose:
            print(f"  MAP: {len(datos)} entradas → {len(pares_intermedios)} pares intermedios")

        if self.combiner:
            pares_intermedios = self._fase_combine(pares_intermedios)

        return pares_intermedios

    def _fase_combine(self, pares):
        """Fase COMBINE (opcional): mini-reduce local."""
        agrupados = defaultdict(list)
        for k, v in pares:
            agrupados[k].append(v)

        combinados = []
        for key, values in agrupados.items():
            resultado = self.combiner(key, values)
            combinados.extend(resultado)

        if self.verbose:
            print(f"  COMBINE: {len(pares)} pares → {len(combinados)} pares (reducción: {(1-len(combinados)/max(len(pares),1))*100:.0f}%)")

        return combinados

    def _fase_shuffle_sort(self, pares):
        """Fase SHUFFLE/SORT: agrupa por clave y ordena."""
        agrupados = defaultdict(list)
        for key, value in pares:
            agrupados[key].append(value)

        resultado = dict(sorted(agrupados.items()))

        if self.verbose:
            print(f"  SHUFFLE/SORT: {len(pares)} pares → {len(resultado)} claves únicas")

        return resultado

    def _fase_reduce(self, agrupados):
        """Fase REDUCE: aplica la función reduce a cada grupo."""
        resultados = {}
        for key, values in agrupados.items():
            resultado = self.reducer(key, values)
            for k, v in resultado:
                resultados[k] = v

        if self.verbose:
            print(f"  REDUCE: {len(agrupados)} claves → {len(resultados)} resultados")

        return resultados

    def ejecutar(self, datos):
        """Ejecuta el pipeline completo: Map → Shuffle/Sort → Reduce."""
        if self.verbose:
            print("=" * 50)
            print("EJECUTANDO MAPREDUCE")
            print("=" * 50)

        inicio = time.time()

        pares_map = self._fase_map(datos)
        agrupados = self._fase_shuffle_sort(pares_map)
        resultados = self._fase_reduce(agrupados)

        tiempo_total = time.time() - inicio

        if self.verbose:
            print(f"  TIEMPO TOTAL: {tiempo_total:.4f}s")
            print("=" * 50)

        return resultados

print("Framework MapReduce cargado.")

---

## 2. Word Count: el "Hello World" de MapReduce

Contar las ocurrencias de cada palabra en un conjunto de textos.

In [ ]:
import re

def wc_mapper(key, value):
    """Emite (palabra, 1) por cada palabra del texto."""
    palabras = re.findall(r'\b\w+\b', value.lower())
    return [(palabra, 1) for palabra in palabras]

def wc_reducer(key, values):
    """Suma todos los conteos de una palabra."""
    return [(key, sum(values))]

textos = [
    ("doc1", "El gato come pescado y el perro come hueso"),
    ("doc2", "El gato duerme en la cama y el perro duerme en el sofá"),
    ("doc3", "Los datos son el nuevo petróleo y los datos generan valor"),
    ("doc4", "Big data transforma el mundo y el mundo necesita datos"),
]

print("TEXTOS DE ENTRADA:")
for key, texto in textos:
    print(f"  {key}: \"{texto}\"")

mr = MapReduceFramework(wc_mapper, wc_reducer, verbose=True)
resultado = mr.ejecutar(textos)

print("\nRESULTADOS (ordenados por frecuencia):")
for palabra, conteo in sorted(resultado.items(), key=lambda x: -x[1])[:15]:
    barra = "█" * conteo
    print(f"  {palabra:<15} {conteo:>3}  {barra}")

---

## 3. Visualización del flujo Map → Shuffle → Reduce

Veamos paso a paso qué ocurre en cada fase con un ejemplo pequeño.

In [ ]:
textos_ejemplo = [
    ("d1", "hola mundo hola"),
    ("d2", "mundo grande mundo"),
]

print("═" * 60)
print("  VISUALIZACIÓN PASO A PASO")
print("═" * 60)

print("\n📥 ENTRADA:")
for k, v in textos_ejemplo:
    print(f"   ({k}, \"{v}\")")

print("\n🔷 FASE MAP:")
todos_pares = []
for key, value in textos_ejemplo:
    pares = wc_mapper(key, value)
    print(f"   Mapper({key}): {pares}")
    todos_pares.extend(pares)

print(f"\n   Todos los pares intermedios: {todos_pares}")

print("\n🔀 FASE SHUFFLE/SORT:")
agrupado = defaultdict(list)
for k, v in todos_pares:
    agrupado[k].append(v)
agrupado = dict(sorted(agrupado.items()))

for key, values in agrupado.items():
    print(f"   {key:<10} → {values}")

print("\n🔶 FASE REDUCE:")
for key, values in agrupado.items():
    resultado = sum(values)
    print(f"   reduce({key}, {values}) = ({key}, {resultado})")

print("\n📤 RESULTADO FINAL:")
for key, values in agrupado.items():
    print(f"   {key}: {sum(values)}")

---

## 4. MapReduce para cálculo de estadísticas

No todo es word count. Calculemos la temperatura media por ciudad con MapReduce.

In [ ]:
import random

random.seed(42)
ciudades = ["Madrid", "Barcelona", "Sevilla", "Valencia", "Bilbao"]

registros_temp = []
for i in range(200):
    ciudad = random.choice(ciudades)
    base_temp = {"Madrid": 15, "Barcelona": 18, "Sevilla": 22, "Valencia": 19, "Bilbao": 13}
    temp = base_temp[ciudad] + random.gauss(0, 5)
    registros_temp.append((f"reg_{i}", f"{ciudad},{temp:.1f}"))

print("Primeros 10 registros:")
for k, v in registros_temp[:10]:
    print(f"  {v}")

def temp_mapper(key, value):
    ciudad, temp = value.split(",")
    return [(ciudad, (float(temp), 1))]

def temp_reducer(key, values):
    suma_total = sum(v[0] for v in values)
    count_total = sum(v[1] for v in values)
    media = suma_total / count_total
    return [(key, round(media, 2))]

def temp_combiner(key, values):
    """Combiner: suma parcial de temperaturas y conteos."""
    suma = sum(v[0] for v in values)
    count = sum(v[1] for v in values)
    return [(key, (suma, count))]

print("\n--- Sin combiner ---")
mr_sin = MapReduceFramework(temp_mapper, temp_reducer, verbose=True)
resultado_sin = mr_sin.ejecutar(registros_temp)

print("\n--- Con combiner ---")
mr_con = MapReduceFramework(temp_mapper, temp_reducer, combiner=temp_combiner, verbose=True)
resultado_con = mr_con.ejecutar(registros_temp)

print("\nTemperatura media por ciudad:")
print(f"{'Ciudad':<15} {'Sin Combiner':>14} {'Con Combiner':>14}")
print("-" * 45)
for ciudad in sorted(resultado_sin.keys()):
    print(f"{ciudad:<15} {resultado_sin[ciudad]:>12.2f}°C {resultado_con[ciudad]:>12.2f}°C")

---

## 5. MapReduce para máximo por categoría

Encontrar el precio máximo de producto por categoría.

In [ ]:
random.seed(123)
categorias = ["Electrónica", "Ropa", "Alimentación", "Hogar", "Deportes"]
productos = []

for i in range(500):
    cat = random.choice(categorias)
    base_precio = {"Electrónica": 200, "Ropa": 50, "Alimentación": 10, "Hogar": 80, "Deportes": 60}
    precio = max(1, base_precio[cat] + random.gauss(0, base_precio[cat] * 0.5))
    productos.append((f"prod_{i}", f"{cat}|Producto_{i}|{precio:.2f}"))

def max_mapper(key, value):
    partes = value.split("|")
    categoria = partes[0]
    nombre = partes[1]
    precio = float(partes[2])
    return [(categoria, (nombre, precio))]

def max_reducer(key, values):
    max_prod = max(values, key=lambda x: x[1])
    return [(key, f"{max_prod[0]} ({max_prod[1]:.2f}€)")]

mr_max = MapReduceFramework(max_mapper, max_reducer, verbose=True)
resultado_max = mr_max.ejecutar(productos)

print("\nProducto más caro por categoría:")
print(f"{'Categoría':<18} {'Producto más caro'}")
print("-" * 50)
for cat, prod in sorted(resultado_max.items()):
    print(f"{cat:<18} {prod}")

---

## 6. Comparación gráfica: impacto del combiner

Midamos cuántos datos se transfieren con y sin combiner.

In [ ]:
import matplotlib.pyplot as plt

texto_grande = " ".join([random.choice(["big", "data", "hadoop", "mapreduce", "spark",
                                         "python", "java", "cluster", "nodo", "bloque",
                                         "el", "la", "de", "en", "y", "un", "una",
                                         "es", "con", "por", "para", "que", "no"])
                         for _ in range(10_000)])

datos_grandes = []
chunk = len(texto_grande) // 10
for i in range(10):
    datos_grandes.append((f"chunk_{i}", texto_grande[i*chunk:(i+1)*chunk]))

todos_pares = []
for key, value in datos_grandes:
    todos_pares.extend(wc_mapper(key, value))
pares_sin_combiner = len(todos_pares)

agrupado_local = defaultdict(int)
for k, v in todos_pares:
    agrupado_local[k] += v
pares_con_combiner = len(agrupado_local)

fig, ax = plt.subplots(figsize=(8, 5))

etiquetas = ['Sin Combiner', 'Con Combiner']
valores = [pares_sin_combiner, pares_con_combiner]
colores = ['#e74c3c', '#2ecc71']

bars = ax.bar(etiquetas, valores, color=colores, width=0.5)

for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{val:,}', ha='center', fontsize=12, fontweight='bold')

ax.set_ylabel('Pares clave-valor transferidos por red', fontsize=11)
ax.set_title('Impacto del Combiner en transferencia de datos', fontsize=13)
reduccion = (1 - pares_con_combiner / pares_sin_combiner) * 100
ax.text(0.5, max(valores) * 0.7, f'Reducción: {reduccion:.0f}%',
        ha='center', fontsize=16, fontweight='bold', color='#2c3e50',
        transform=ax.transAxes if False else ax.transData)

plt.tight_layout()
plt.show()

print(f"Sin combiner: {pares_sin_combiner:,} pares por red")
print(f"Con combiner: {pares_con_combiner:,} pares por red")
print(f"Reducción del tráfico de red: {reduccion:.1f}%")

---

## Recursos

- [MapReduce Paper (2004)](https://static.googleusercontent.com/media/research.google.com/en//archive/mapreduce-osdi04.pdf)
- [MapReduce Tutorial - Apache Hadoop](https://hadoop.apache.org/docs/stable/hadoop-mapreduce-client/hadoop-mapreduce-client-core/MapReduceTutorial.html)
- [Python functools.reduce](https://docs.python.org/3/library/functools.html#functools.reduce)
- [MapReduce Design Patterns](https://www.oreilly.com/library/view/mapreduce-design-patterns/9781449341954/)